In [20]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

sns.set_style("whitegrid")

In [21]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)


In [22]:
# Load Dataset from UCI
# Fetch dataset
heart_disease = fetch_ucirepo(id=45)
# Features and target
X = heart_disease.data.features
y = heart_disease.data.targets
df = pd.concat([X, y], axis=1)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Create binary target
df["target"] = (df["num"] > 0).astype(int)

# Features and target
X = df.drop(columns=["num", "target"])
y = df["target"]

# All features are numeric
numeric_features = X.columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        )
    ]
)


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)

(242, 13)
(61, 13)


In [33]:
# Configure MLflow
mlflow.set_experiment("Heart Disease Prediction")

2026/10/19 20:30:07 INFO mlflow.tracking.fluent: Experiment with name 'Heart Disease Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/Sandeep/Documents/heart-disease-mlops/notebooks/mlruns/1', creation_time=1792422007060, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792422007060, lifecycle_stage='active', name='Heart Disease Prediction', tags={}, trace_location=None, workspace='default'>

In [17]:
# Logistic Regression Cell

In [35]:
with mlflow.start_run(run_name="LogisticRegression"):

    logistic_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    logistic_model.fit(X_train, y_train)

    preds = logistic_model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, preds)

    cv_scores = cross_val_score(
        logistic_model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    mlflow.log_param("model", "LogisticRegression")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("cv_accuracy", cv_scores.mean())

    mlflow.sklearn.log_model(
        logistic_model,
        name="logistic_model",
        serialization_format="cloudpickle"
    )

    print("LOGISTIC REGRESSION")

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC AUC  :", roc)
    print("CV Mean  :", cv_scores.mean())

    print(classification_report(y_test, preds))

    logistic_accuracy = accuracy

2026/10/19 20:31:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LOGISTIC REGRESSION
Accuracy : 0.8688524590163934
Precision: 0.8125
Recall   : 0.9285714285714286
F1 Score : 0.8666666666666666
ROC AUC  : 0.8733766233766233
CV Mean  : 0.8282513661202184
              precision    recall  f1-score   support

           0       0.93      0.82      0.87        33
           1       0.81      0.93      0.87        28

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.88      0.87      0.87        61



In [ ]:
# Random Forest

In [37]:
with mlflow.start_run(run_name="RandomForest"):

    rf_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            random_state=42
        ))
    ])

    rf_model.fit(X_train, y_train)

    preds = rf_model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, preds)

    cv_scores = cross_val_score(
        rf_model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("cv_accuracy", cv_scores.mean())

    mlflow.sklearn.log_model(
        rf_model,
        name="random_forest_model",
        serialization_format="cloudpickle"
    )

    print("RANDOM FOREST")

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC AUC  :", roc)
    print("CV Mean  :", cv_scores.mean())

    print(classification_report(y_test, preds))

    rf_accuracy = accuracy

2026/10/19 20:32:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RANDOM FOREST
Accuracy : 0.9016393442622951
Precision: 0.8666666666666667
Recall   : 0.9285714285714286
F1 Score : 0.896551724137931
ROC AUC  : 0.9036796536796536
CV Mean  : 0.8248087431693989
              precision    recall  f1-score   support

           0       0.94      0.88      0.91        33
           1       0.87      0.93      0.90        28

    accuracy                           0.90        61
   macro avg       0.90      0.90      0.90        61
weighted avg       0.90      0.90      0.90        61



In [ ]:
# Model Comparison

In [39]:
comparison = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Random Forest"
        ],
        "Accuracy": [
            logistic_accuracy,
            rf_accuracy
        ]
    }
)

comparison.sort_values(
    by="Accuracy",
    ascending=False
)


,Model,Accuracy
1,Random Forest,0.901639
0,Logistic Regression,0.868852


In [ ]:
# Save Best Model

In [41]:
if rf_accuracy > logistic_accuracy:

    best_model = rf_model
    best_name = "Random Forest"

else:

    best_model = logistic_model
    best_name = "Logistic Regression"

joblib.dump(
    best_model,
    "../models/model.pkl"
)

print(f"Best Model Saved: {best_name}")

Best Model Saved: Random Forest
